# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
display_name = metadata.name if hasattr(metadata, 'name') else getattr(metadata, 'name', None)
display_description = metadata.description if hasattr(metadata, 'description') else getattr(metadata, 'description', None)
print(f"{display_name}: {display_description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In Croissant datasets, **record sets** are central containers for tabular or structured data. Each record set has an `@id` you can query and typically contains several **fields** (columns), also referenced by their `@id`.

In [ ]:
# List record sets and their fields by @id

# Croissant schemas expose record sets in dataset.metadata.record_sets,
# but as mlcroissant mirrors the JSON-LD structure, the attribute is usually 'record_sets' or 'recordSet'.
# For this dataset, let's try both attribute names:

def show_recordsets_and_fields(metadata):
    # Try both possible attribute names (record_sets, recordSet)
    record_sets = []
    if hasattr(metadata, 'record_sets') and metadata.record_sets:
        record_sets = metadata.record_sets
    elif hasattr(metadata, 'recordSet') and metadata.recordSet:
        record_sets = metadata.recordSet
    else:
        print('No record sets found in the Croissant metadata.')
        return
    print('Available Record Sets:')
    for rs in record_sets:
        print(f"  Record set name: {rs.name if hasattr(rs, 'name') else getattr(rs, 'name', None)} | @id: {rs.id if hasattr(rs, 'id') else getattr(rs, 'id', None)}")
        # List fields
        # Try possible attributes for fields
        fields = []
        if hasattr(rs, 'fields') and rs.fields:
            fields = rs.fields
        elif hasattr(rs, 'field') and rs.field:
            fields = rs.field
        if fields:
            for fld in fields:
                print(f"     Field: {fld.name if hasattr(fld, 'name') else getattr(fld, 'name', None)} | @id: {fld.id if hasattr(fld, 'id') else getattr(fld, 'id', None)}")
    return record_sets

record_sets = show_recordsets_and_fields(metadata)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

_Note: For this dataset, we'll extract all available record sets, referencing each by its `@id`._

In [ ]:
# Extract data from each record set

# Collect all record set @ids (using .id for mlcroissant objects)
record_set_ids = []

def get_record_set_ids(metadata):
    ids = []
    record_sets = []
    if hasattr(metadata, 'record_sets') and metadata.record_sets:
        record_sets = metadata.record_sets
    elif hasattr(metadata, 'recordSet') and metadata.recordSet:
        record_sets = metadata.recordSet
    for rs in record_sets:
        rsid = rs.id if hasattr(rs, 'id') else getattr(rs, 'id', None)
        if rsid is not None:
            ids.append(rsid)
    return ids

record_set_ids = get_record_set_ids(metadata)
print(f"Record sets found: {record_set_ids}")

dataframes = {}
for record_set in record_set_ids:
    print(f'Extracting records from record set @id: {record_set}')
    try:
        records = list(dataset.records(record_set=record_set))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set] = df
            print(f"Loaded DataFrame for {record_set}. Columns: {df.columns.tolist()}")
            display(df.head())
        else:
            print(f"No records found for {record_set}.")
    except Exception as e:
        print(f"Failed to load records for record set {record_set}: {e}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

We'll demonstrate EDA for the **first available record set** (adjust the `record_set_id` and field IDs as needed based on the above output).

In [ ]:
# Configuration for EDA
import numpy as np

# Pick the first available record set
if len(dataframes) > 0:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    print(f'Using record set @id: {record_set_id}')
else:
    print("No record sets available for EDA.")

# Identify a numeric field (column) in the DataFrame
numeric_field = None
for col in df.columns:
    if np.issubdtype(df[col].dtype, np.number):
        numeric_field = col
        break
if numeric_field:
    print(f"Selected numeric field: {numeric_field}")
    threshold = df[numeric_field].mean() if not np.isnan(df[numeric_field].mean()) else 0
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    display(filtered_df.head())

    # Normalize the selected numeric column
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by a categorical field if present
    group_field = None
    for col in df.columns:
        if df[col].dtype == object and col != numeric_field:
            group_field = col
            break
    if group_field:
        print(f'Grouping by field: {group_field}')
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f'Grouped data by {group_field}:')
        display(grouped_df.head())
else:
    print('No numeric fields found for EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the numeric field's distribution and normalized values
if len(dataframes) > 0 and numeric_field is not None:
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    sns.histplot(df[numeric_field], kde=True)
    plt.title(f'Original Distribution: {numeric_field}')
    plt.tight_layout()
    if f"{numeric_field}_normalized" in filtered_df.columns:
        plt.subplot(1, 2, 2)
        sns.histplot(filtered_df[f"{numeric_field}_normalized"], kde=True, color='orange')
        plt.title(f'Normalized: {numeric_field}')
    plt.tight_layout()
    plt.show()

    # If group_field exists, plot group means
    if group_field is not None and group_field in df.columns:
        plt.figure(figsize=(10,4))
        order = None
        try:
            order = grouped_df.sort_values(numeric_field, ascending=False)[group_field]
        except:
            order = None
        sns.barplot(data=grouped_df, x=group_field, y=numeric_field, order=order)
        plt.title(f'Mean {numeric_field} by {group_field}')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print('No suitable data found for visualization.')

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

*In this notebook, we demonstrated the use of `mlcroissant` to load and explore a Croissant-described dataset. We reviewed available record sets and fields, extracted tabular data, and performed basic exploratory analysis and visualization. For more advanced analyses, consult the full Croissant schema and consider referencing fields and record sets by their `@id` as illustrated. This approach maximizes future-proofing and interoperability for FAIR data workflows.*